# Pipeline A — YOLO26: Demo Notebook

This notebook walks through the three Pipeline A steps:
1. `tile_dataset.py` — prepare tiled training data
2. `train_yolo.py` — fine-tune YOLO26-seg on dendrites
3. `predict_tiled.py` — run tiled inference on test images

**Folder structure assumed:**
```
root/
  notebooks/          ← you are here
  pipeline_A_yolo/
    tile_dataset.py
    train_yolo.py
    predict_tiled.py
  data/
    annotated/        ← Roboflow export (no tiling)
      train/images/
      train/labels/
      val/images/
      val/labels/
      test/images/
      test/labels/
      data.yaml
    tiled/            ← created by tile_dataset.py
  outputs/
    weights/          ← best.pt saved here after training
    masks/            ← prediction masks saved here
    visuals/
```

## 0. Setup

In [ ]:
import sys
import os
from pathlib import Path

# ── PATH SETUP ───────────────────────────────────────────────────────────────
# We're inside notebooks/ so we go one level up to reach the project root
ROOT = Path('..').resolve()
PIPELINE_A_DIR = ROOT / 'pipeline_A_yolo'

# Add pipeline_A_yolo to path so we can import from it
sys.path.insert(0, str(PIPELINE_A_DIR))
sys.path.insert(0, str(ROOT))

print(f'Project root : {ROOT}')
print(f'Pipeline A   : {PIPELINE_A_DIR}')
assert PIPELINE_A_DIR.exists(), f'pipeline_A_yolo/ not found at {PIPELINE_A_DIR}'

# ── STANDARD IMPORTS ─────────────────────────────────────────────────────────
import cv2
import numpy as np
import matplotlib.pyplot as plt
import yaml

# ── PIPELINE A IMPORTS ───────────────────────────────────────────────────────
from tile_dataset   import make_tiled_dataset, get_tile_coords
from predict  import predict_single_image, visualize_prediction
# train_yolo is run as a script (see Section 2) not imported directly

print('\nAll imports OK')

In [ ]:
# ── CONFIGURE PATHS HERE ─────────────────────────────────────────────────────
ANNOTATED_DIR = ROOT / 'data' / 'annotated'   # Roboflow export root
TILED_DIR     = ROOT / 'data' / 'tiled'       # output of tile_dataset.py
WEIGHTS_DIR   = ROOT / 'outputs' / 'weights'
MASKS_DIR     = ROOT / 'outputs' / 'masks'
VISUALS_DIR   = ROOT / 'outputs' / 'visuals'

DATA_YAML     = ANNOTATED_DIR / 'data.yaml'
TILED_YAML    = TILED_DIR / 'data.yaml'
BEST_WEIGHTS  = WEIGHTS_DIR / 'best.pt'

# Verify annotated data exists before proceeding
assert ANNOTATED_DIR.exists(), \
    f'Roboflow export not found at {ANNOTATED_DIR}\nDownload and extract it first.'
assert DATA_YAML.exists(), \
    f'data.yaml not found at {DATA_YAML}'

# Print dataset summary
with open(DATA_YAML) as f:
    yaml_data = yaml.safe_load(f)

print('Dataset summary:')
for split in ['train', 'val', 'test']:
    split_path = ANNOTATED_DIR / split / 'images'
    if split_path.exists():
        n = len(list(split_path.glob('*')))
        print(f'  {split:6s}: {n} images')
print(f'  Classes: {yaml_data.get("names", "not found")}')

## 1. Tile Dataset
Tiles the train split (12 images → ~70 tiles) and copies val/test as-is.

**Skip this cell if `data/tiled/` already exists and looks correct.**

In [ ]:
# ── TILING PARAMETERS ────────────────────────────────────────────────────────
TILE_SIZE = 640    # must match imgsz in train_yolo.py
OVERLAP   = 0.2    # must match overlap in predict_tiled.py
# ─────────────────────────────────────────────────────────────────────────────

if TILED_DIR.exists():
    existing_tiles = list((TILED_DIR / 'train' / 'images').glob('*.png'))
    print(f'Tiled dataset already exists: {len(existing_tiles)} train tiles found')
    print('Delete data/tiled/ and re-run this cell if you want to re-tile')
else:
    print('Running tile_dataset.py...')
    make_tiled_dataset(
        input_dir=str(ANNOTATED_DIR),
        output_dir=str(TILED_DIR),
        tile_size=TILE_SIZE,
        overlap=OVERLAP
    )

In [ ]:
# ── VERIFY TILING RESULTS ────────────────────────────────────────────────────
print('Tiled dataset contents:')
for split in ['train', 'val', 'test']:
    img_dir = TILED_DIR / split / 'images'
    lbl_dir = TILED_DIR / split / 'labels'
    if img_dir.exists():
        n_imgs = len(list(img_dir.glob('*')))
        n_lbls = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
        n_annotated = sum(
            1 for l in lbl_dir.glob('*.txt') if l.stat().st_size > 0
        ) if lbl_dir.exists() else 0
        print(f'  {split:6s}: {n_imgs:4d} images | '
              f'{n_lbls:4d} labels | '
              f'{n_annotated:4d} with annotations')

In [ ]:
# ── VISUALIZE A FEW TILES ─────────────────────────────────────────────────────
# Pick a few training tiles and show them with their annotations overlaid

def draw_yolo_polygons(image, label_path, color=(0, 255, 0), thickness=2):
    """Draw YOLO polygon annotations on an image."""
    h, w = image.shape[:2]
    annotated = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR) \
                if len(image.shape) == 2 else image.copy()

    if not Path(label_path).exists() or Path(label_path).stat().st_size == 0:
        return annotated  # empty label

    with open(label_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            coords = parts[1:]  # skip class_id
            points = []
            for i in range(0, len(coords), 2):
                px = int(coords[i]   * w)
                py = int(coords[i+1] * h)
                points.append([px, py])
            pts = np.array(points, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(annotated, [pts], isClosed=True, color=color, thickness=thickness)

    return annotated


# Find tiles that have annotations
train_tiles = sorted((TILED_DIR / 'train' / 'images').glob('*.png'))
annotated_tiles = [
    t for t in train_tiles
    if (TILED_DIR / 'train' / 'labels' / (t.stem + '.txt')).stat().st_size > 0
]

# Show up to 6 annotated tiles
show_tiles = annotated_tiles[:6]
n = len(show_tiles)

if n == 0:
    print('No annotated tiles found — check tile_dataset.py output')
else:
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, tile_path in zip(axes, show_tiles):
        img = cv2.imread(str(tile_path))
        lbl = TILED_DIR / 'train' / 'labels' / (tile_path.stem + '.txt')
        annotated = draw_yolo_polygons(img, lbl)
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(tile_path.stem[-20:], fontsize=8)
        ax.axis('off')

    plt.suptitle(f'Sample Annotated Tiles ({n} shown)', fontsize=13)
    plt.tight_layout()
    plt.show()
    print(f'Total annotated tiles: {len(annotated_tiles)} / {len(train_tiles)}')

## 2. Train YOLO26-seg
Runs `train_yolo.py` as a subprocess so output streams live to the notebook.

**Skip this section if `outputs/weights/best.pt` already exists.**

Expected time: 5-15 min (GPU) / 15-30 min (Apple Silicon) / 45-90 min (CPU)

In [ ]:
# ── CHECK IF WEIGHTS ALREADY EXIST ───────────────────────────────────────────
if BEST_WEIGHTS.exists():
    print(f'Weights already exist: {BEST_WEIGHTS}')
    print('Skip to Section 3 for inference.')
    print('Delete outputs/weights/best.pt to retrain.')
else:
    print('No weights found — run the training cell below.')

In [ ]:
# ── TRAINING PARAMETERS ───────────────────────────────────────────────────────
# Adjust these if needed, then run this cell to start training
MODEL   = 'yolo12n-seg.pt'   # change to yolo26n-seg.pt when available
EPOCHS  = 100
IMGSZ   = TILE_SIZE          # must match tile size above
BATCH   = -1                 # -1 = auto
LR      = 0.001
# ─────────────────────────────────────────────────────────────────────────────

import subprocess

train_cmd = [
    sys.executable,
    str(PIPELINE_A_DIR / 'train_yolo.py'),
    '--data',    str(TILED_YAML),
    '--model',   MODEL,
    '--epochs',  str(EPOCHS),
    '--imgsz',   str(IMGSZ),
    '--batch',   str(BATCH),
    '--lr',      str(LR),
    '--output',  str(WEIGHTS_DIR),
]

print('Starting training...')
print('Command:', ' '.join(train_cmd))
print('-' * 55)

# Stream output live to notebook
process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    print(line, end='')

process.wait()
print('-' * 55)
print(f'Training finished. Exit code: {process.returncode}')

In [ ]:
# ── PLOT TRAINING CURVES ──────────────────────────────────────────────────────
# YOLO saves a results.csv in the run directory with per-epoch metrics
# Find the most recent run directory

import pandas as pd

run_dirs = sorted(WEIGHTS_DIR.glob('dendrite_*'), key=lambda p: p.stat().st_mtime)

if not run_dirs:
    print('No training run directories found yet. Run training first.')
else:
    run_dir = run_dirs[-1]
    results_csv = run_dir / 'results.csv'

    if not results_csv.exists():
        print(f'results.csv not found in {run_dir}')
    else:
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()  # remove whitespace from headers

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Loss curves
        for col, label in [('train/seg_loss', 'Train seg loss'),
                            ('val/seg_loss',   'Val seg loss')]:
            if col in df.columns:
                axes[0].plot(df['epoch'], df[col], label=label)
        axes[0].set_title('Segmentation Loss')
        axes[0].set_xlabel('Epoch')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # mAP curves
        for col, label in [('metrics/mAP50(M)',    'mAP@50 (seg)'),
                            ('metrics/mAP50-95(M)', 'mAP@50-95 (seg)')]:
            if col in df.columns:
                axes[1].plot(df['epoch'], df[col], label=label)
        axes[1].set_title('Segmentation mAP')
        axes[1].set_xlabel('Epoch')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

        # Precision / Recall
        for col, label in [('metrics/precision(M)', 'Precision'),
                            ('metrics/recall(M)',    'Recall')]:
            if col in df.columns:
                axes[2].plot(df['epoch'], df[col], label=label)
        axes[2].set_title('Precision & Recall')
        axes[2].set_xlabel('Epoch')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)

        plt.suptitle(f'Training Results — {run_dir.name}', fontsize=13)
        plt.tight_layout()

        viz_path = VISUALS_DIR / 'training_curves.png'
        VISUALS_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(str(viz_path), bbox_inches='tight', dpi=150)
        plt.show()
        print(f'Saved: {viz_path}')

## 3. Predict on Test Images
Runs tiled inference on the 5 test images using `predict_tiled.py`.

In [ ]:
# ── VERIFY WEIGHTS EXIST ─────────────────────────────────────────────────────
assert BEST_WEIGHTS.exists(), \
    f'Weights not found at {BEST_WEIGHTS}\nRun Section 2 first.'

from ultralytics import YOLO
model = YOLO(str(BEST_WEIGHTS))
print(f'Model loaded from {BEST_WEIGHTS}')

# Test images
test_img_dir = TILED_DIR / 'test' / 'images'
test_images  = sorted(test_img_dir.glob('*.png')) + \
               sorted(test_img_dir.glob('*.jpg'))

print(f'Test images found: {len(test_images)}')
for p in test_images:
    print(f'  {p.name}')

In [ ]:
# ── INFERENCE PARAMETERS ─────────────────────────────────────────────────────
CONF_THRESHOLD = 0.25   # try: 0.15, 0.25, 0.35
IOU_THRESHOLD  = 0.45   # NMS threshold, usually fine at default
# ─────────────────────────────────────────────────────────────────────────────

# Run prediction on all test images
MASKS_DIR.mkdir(parents=True, exist_ok=True)
predicted_masks = {}

for img_path in test_images:
    print(f'Predicting: {img_path.name}...', end=' ')

    mask = predict_single_image(
        model=model,
        image_path=str(img_path),
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        conf_threshold=CONF_THRESHOLD,
        iou_threshold=IOU_THRESHOLD
    )

    mask_path = MASKS_DIR / f'{img_path.stem}_yolo_mask.png'
    cv2.imwrite(str(mask_path), mask)
    predicted_masks[img_path.name] = mask

    coverage = (mask > 0).mean() * 100
    print(f'done  ({coverage:.1f}% dendrite coverage)')

print(f'\nAll masks saved to {MASKS_DIR}')

In [ ]:
# ── VISUALIZE PREDICTIONS ────────────────────────────────────────────────────
# Show original + mask + overlay for each test image

n_imgs = len(test_images)
if n_imgs == 0:
    print('No test images found')
else:
    fig, axes = plt.subplots(n_imgs, 3, figsize=(18, 6 * n_imgs))

    # Handle single image case
    if n_imgs == 1:
        axes = [axes]

    for row_axes, img_path in zip(axes, test_images):
        original = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        mask     = predicted_masks[img_path.name]

        # Create overlay
        rgb     = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB)
        overlay = rgb.copy()
        overlay[mask > 0] = [0, 200, 0]
        blended = cv2.addWeighted(rgb, 0.6, overlay, 0.4, 0)

        row_axes[0].imshow(original, cmap='gray')
        row_axes[0].set_title(f'{img_path.name}\nOriginal', fontsize=10)
        row_axes[0].axis('off')

        row_axes[1].imshow(mask, cmap='gray')
        row_axes[1].set_title('YOLO Mask', fontsize=10)
        row_axes[1].axis('off')

        row_axes[2].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
        row_axes[2].set_title('Overlay (green = dendrite)', fontsize=10)
        row_axes[2].axis('off')

    plt.suptitle('YOLO26 Predictions — Test Set', fontsize=14, y=1.01)
    plt.tight_layout()

    save_path = VISUALS_DIR / 'yolo_test_predictions.png'
    plt.savefig(str(save_path), bbox_inches='tight', dpi=150)
    plt.show()
    print(f'Saved: {save_path}')

In [ ]:
# ── TUNE CONFIDENCE THRESHOLD INTERACTIVELY ───────────────────────────────────
# Run this cell to see how different confidence thresholds affect one image
# Pick the image you want to inspect

INSPECT_IMAGE = test_images[0]   # change index to inspect a different image
THRESHOLDS    = [0.10, 0.20, 0.30, 0.40]

fig, axes = plt.subplots(1, len(THRESHOLDS) + 1, figsize=(5 * (len(THRESHOLDS) + 1), 5))

original = cv2.imread(str(INSPECT_IMAGE), cv2.IMREAD_GRAYSCALE)
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

for ax, thresh in zip(axes[1:], THRESHOLDS):
    mask = predict_single_image(
        model=model,
        image_path=str(INSPECT_IMAGE),
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
        conf_threshold=thresh
    )
    coverage = (mask > 0).mean() * 100
    ax.imshow(mask, cmap='gray')
    ax.set_title(f'conf={thresh}\n{coverage:.1f}% coverage', fontsize=10)
    ax.axis('off')

plt.suptitle(f'Confidence Threshold Comparison — {INSPECT_IMAGE.name}', fontsize=12)
plt.tight_layout()
plt.show()
print('Pick the threshold that best captures dendrites without too much background noise')
print(f'Then update CONF_THRESHOLD above and re-run the prediction cell')